# Train

Trains a single model on the full screened universe using per-symbol temporal splits.
Evaluates on two held-out sets:

1. **Temporal test** — post-cutoff windows of training symbols
2. **Held-out symbols** — symbols never seen during training

Prerequisites:
- `data/symbols.yml` — screened universe (produced by `screen_by_coverage`)
- `data/historical-prices/<SYMBOL>/<YEAR>.csv` — daily OHLCV
- `data/sentiment/<SYMBOL>.parquet` — pre-encoded daily sentiment (optional)

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import yaml

import src  # triggers load_dotenv()
from src.log import setup_logging

setup_logging()
logger = logging.getLogger("train")

## Config

In [ ]:
from src.training import ComputeConfig, TrainingConfig, Split

MODEL_TYPE      = "lstm"   # "lstm" or "transformer"
CHECKPOINT_NAME = f"universal_{MODEL_TYPE}_v1"
SPLIT_PATH      = Path("../data/splits.yml")
PRICE_YEARS     = [2018, 2019, 2020]

compute_config = ComputeConfig(num_workers=0)
compute_config.setup()

config = TrainingConfig(
    window=64,
    batch_size=32,
    n_epochs=100,
    lr=1e-3,
    patience=15,
    dropout=0.2,
)

print(f"Device : {compute_config.device}")
print(f"Model  : {MODEL_TYPE}")
print(f"Window : {config.window}")

## Load Universe + Split

If `data/splits.yml` exists it is loaded directly; otherwise a new split is
created from `data/symbols.yml` and saved for reproducibility.

In [ ]:
if SPLIT_PATH.exists():
    split = Split.load(SPLIT_PATH)
    print(f"Loaded split from {SPLIT_PATH}")
else:
    symbols = list(
        yaml.safe_load(Path("../data/symbols.yml").read_text())["companies"].keys()
    )
    split = Split.create(
        symbols,
        held_out_frac=0.2,
        nominal_cutoff="2019-10-01",
        stagger_days=45,
        val_months=2,
        seed=42,
    )
    split.save(SPLIT_PATH)
    print(f"Created new split, saved to {SPLIT_PATH}")

print(f"Train symbols   : {len(split.train_symbols)}")
print(f"Held-out symbols: {len(split.held_out_symbols)}")

## Load Price Data

In [ ]:
from src.repositories.prices import PriceRepository

prices = PriceRepository()
price_dfs: dict = {}

for symbol in split.all_symbols:
    try:
        price_dfs[symbol] = prices.load_years(symbol, PRICE_YEARS)
    except FileNotFoundError:
        price_dfs[symbol] = None

n_ok = sum(1 for v in price_dfs.values() if v is not None)
print(f"Price data loaded: {n_ok} / {len(split.all_symbols)} symbols")

## Load Sentiment

Symbols without a cached parquet use zero vectors for sentiment inputs.

In [ ]:
from src.repositories.sentiment import SentimentRepository

sent_repo = SentimentRepository()
sentiment_dfs: dict = {}

for symbol in split.all_symbols:
    if sent_repo.exists(symbol):
        sentiment_dfs[symbol] = sent_repo.load(symbol)
    else:
        sentiment_dfs[symbol] = None

n_ok = sum(1 for v in sentiment_dfs.values() if v is not None)
print(f"Sentiment loaded: {n_ok} / {len(split.all_symbols)} symbols")

## Build Datasets

One `StockDataset` per symbol. Symbols with insufficient price history are skipped.

In [ ]:
from src.features.dataset import StockDataset

datasets: dict[str, StockDataset] = {}
skipped: list[str] = []

for symbol in split.all_symbols:
    df = price_dfs.get(symbol)
    if df is None:
        skipped.append(symbol)
        continue
    try:
        datasets[symbol] = StockDataset(
            symbol=symbol,
            price_df=df,
            sentiment_df=sentiment_dfs.get(symbol),
            window=config.window,
        )
    except RuntimeError as exc:
        logger.warning("Skipping %s: %s", symbol, exc)
        skipped.append(symbol)

print(f"Datasets built: {len(datasets)} / {len(split.all_symbols)}")
if skipped:
    print(f"Skipped ({len(skipped)}): {skipped[:10]}{'...' if len(skipped) > 10 else ''}")

## Build DataLoaders

`DataLoaderBuilder` applies per-symbol temporal splits, fits scalers on training
data, and returns `(train_loader, val_loader, test_loader)`.

In [ ]:
from src.features.dataset import DataLoaderBuilder

builder = DataLoaderBuilder(datasets, split, config, compute_config)
train_loader, val_loader, test_loader = builder.build()

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

## Model

In [ ]:
from src.model.lstm import SentimentLSTM
from src.model.transformer import SentimentTransformer

n_sentiment_probs = builder.n_sentiment_probs

if MODEL_TYPE == "lstm":
    model = SentimentLSTM(
        n_factors=16,
        sentiment_dim=768,
        hidden_size=32,
        num_layers=2,
        dropout=config.dropout,
        n_sentiment_probs=n_sentiment_probs,
    )
else:
    model = SentimentTransformer(
        n_factors=16,
        sentiment_dim=768,
        d_model=64,
        nhead=4,
        n_layers=6,
        dim_feedforward=128,
        dropout=config.dropout,
        n_sentiment_probs=n_sentiment_probs,
    )

print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

## Train

In [ ]:
from src.model.trainer import Trainer

trainer = Trainer(model, config, compute_config)
result = trainer.fit(train_loader, val_loader)

print(f"Best epoch: {result.best_epoch}  |  Best val AUC: {result.best_val_auc:.4f}")

## Evaluate — Temporal Test Set

Post-cutoff windows of training symbols: the model saw these stocks during training
but on different (earlier) time windows.

In [ ]:
r = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=42)

print("=== Temporal test set (training symbols, held-out time) ===")
print(f"AUC      : {r.auc_mean:.3f}  [{r.auc_ci_low:.3f}, {r.auc_ci_high:.3f}]")
print(f"Accuracy : {r.accuracy_mean:.3f}  [{r.accuracy_ci_low:.3f}, {r.accuracy_ci_high:.3f}]")
print(f"Precision: {r.precision_mean:.3f}  [{r.precision_ci_low:.3f}, {r.precision_ci_high:.3f}]")
print(f"Recall   : {r.recall_mean:.3f}  [{r.recall_ci_low:.3f}, {r.recall_ci_high:.3f}]")

## Evaluate — Held-Out Symbols

Symbols never seen during training — tests cross-symbol generalisation.

In [ ]:
held_out_loader = builder.build_held_out_loader()
r_ho = trainer.bootstrap_evaluate(held_out_loader, n_bootstrap=1000, seed=42)

print("=== Held-out symbols (never seen during training) ===")
print(f"AUC      : {r_ho.auc_mean:.3f}  [{r_ho.auc_ci_low:.3f}, {r_ho.auc_ci_high:.3f}]")
print(f"Accuracy : {r_ho.accuracy_mean:.3f}  [{r_ho.accuracy_ci_low:.3f}, {r_ho.accuracy_ci_high:.3f}]")
print(f"Precision: {r_ho.precision_mean:.3f}  [{r_ho.precision_ci_low:.3f}, {r_ho.precision_ci_high:.3f}]")
print(f"Recall   : {r_ho.recall_mean:.3f}  [{r_ho.recall_ci_low:.3f}, {r_ho.recall_ci_high:.3f}]")

## Save Checkpoint

In [ ]:
from src.repositories.models import ModelRepository

model_repo = ModelRepository()
model_repo.save(
    CHECKPOINT_NAME,
    model,
    builder.tech_scaler,
    {
        "model_type": MODEL_TYPE,
        "window": config.window,
        "n_sentiment_probs": n_sentiment_probs,
        "training_history": result.history,
        "best_epoch": result.best_epoch,
        "best_val_auc": result.best_val_auc,
        "temporal_test_auc": r.auc_mean,
        "held_out_auc": r_ho.auc_mean,
    },
)
print(f"Saved: {CHECKPOINT_NAME}")
print(f"Available checkpoints: {model_repo.list()}")